> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [エージェント概要](#エージェント概要)
- [ModelRouterAgentの作成](#modelrouteragentの作成)
- [FileSearchAgentの作成](#filesearchagentの作成)
- [WebSearchAgentの作成](#websearchagentの作成)
- [エージェントのデプロイと呼び出し](#エージェントのデプロイと呼び出し)

## 🎯 学習目標

- Microsoft Foundryエージェントのコア概念の理解
- Model Routerベースのエージェント構築
- File Search機能を活用したドキュメントベースのエージェント作成
- Web Search機能を活用したリアルタイム情報検索エージェント作成
- エージェントのデプロイとプログラマティック呼び出し方法の学習

## ⏱️ 予想所要時間

約30分

## 環境設定

まず プロジェクト エンドポイントを 設定します.

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLIを 見つを 数 あるも録)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前 ノートブックで 保存した 設定ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (異なる もそれらが 使用する 数 あるも録)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました。")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つを 数 ありません。")
    print("💡 01-setup.ipynbを まず 実行して 環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-projects==2.0.0b2 azure-identity

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用する プロジェクト エンドポイント: {PROJECT_ENDPOINT}")

## ModelRouterAgentの作成

Model Routerを 活用して インテリジェントに モデルを 選択するは エージェントを 作成します.

**Agent 構成:**
- **Model**: model-router (費用/品質/パフォーマンス 自動 最適化)
- **Instructions**: 質問 回答 エージェント
- **Tools**: なし (デフォルト 対話)

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ModelRouterAgent 作成
ROUTER_INSTRUCTIONS = """あなたは 質問に 回答するは エージェントです.
リクエストの 複雑もと 要件に に従って が章 適切な モデルを 使用してください.
常に 名確で 正確し も動きが なりは 回答を 提供してください."""

try:
    definition = PromptAgentDefinition(
        model="model-router",
        instructions=ROUTER_INSTRUCTIONS
    )
    
    agent_router = client.agents.create(
        name="ModelRouterAgent",
        definition=definition
    )
    
    print(f"✅ ModelRouterAgent 作成 完了!")
    print(f"   ID: {agent_router.id}")
    print(f"   Name: {agent_router.name}")
    print(f"   Model: {definition.model}")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決 方法:")
    print("   1. Portalで 'model-router' モデルが デプロイなったはない 確認")
    print("   2. モデル 名前が 正確なない 確認 (大文字小文字 区別)")

## FileSearchAgentの作成

ファイル 検索 機能を 活用して アップロードされた ドキュメントで 情報を 見つは エージェントです.

**Agent 構成:**
- **Model**: gpt-5.1
- **Tools**: file_search (ファイル 内容 検索)
- **Files**: knowledge-base.json アップロード

**⚠️ 注意**: ファイル アップロードは Azure Portal(https://ai.azure.com)で より 便利します.
- Build > Agents > Create agent > Tools > File Search > Upload files

In [ ]:
# FileSearchAgent 作成

FILE_SEARCH_INSTRUCTIONS = """あなたは Toolsに 登録された File search ベースで 回答するは エージェントです.

重要 ルール:
1. 必ず アップロードされた ファイルの 内容を ベースでだけ 回答してください
2. ファイルに ないは 情報は "提供された ドキュメントで 該当 情報を 見つを 数 ありません"と 回答してください
3. 回答 時 ソース ファイル名を 言及してください
4. 正確な 引用を 使用してください"""

try:
    # FileSearchAgent 作成 (tools ないが まず 作成)
    definition = PromptAgentDefinition(
        model="gpt-5.1",
        instructions=FILE_SEARCH_INSTRUCTIONS
    )
    
    agent_filesearch = client.agents.create(
        name="FileSearchAgent",
        definition=definition
    )
    
    print(f"✅ FileSearchAgent 作成 完了!")
    print(f"   ID: {agent_filesearch.id}")
    print(f"   Name: {agent_filesearch.name}")
    print(f"   Model: {definition.model}")
    print(f"\n📋 次のステップ: Azure Portalで File Search も旧 追加")
    print(f"   1. Azure Portal (https://ai.azure.com) 接続")
    print(f"   2. Build > Agents > 'FileSearchAgent' 選択")
    print(f"   3. Tools セクションで 'File Search' も旧 追加")
    print(f"   4. knowledge-base.json ファイル アップロード")
    print(f"   5. アップロード 後 エージェントが ファイル 内容を 検索する 数 あります")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決 方法:")
    print("   1. 'gpt-5.1' モデルが デプロイなったはない 確認")
    print("   2. Portalで モデル 名前 確認 (大文字小文字 区別)")

## WebSearchAgentの作成

Web 検索 機能を 活用して リアルタイム 情報を 提供するは エージェントです.

**Agent 構成:**
- **Model**: gpt-4.1
- **Tools**: web_search (Web 検索)
- **機能**: 最新 ニュース, 天気, 株式 情報 など

In [ ]:
# WebSearchAgent 作成

WEB_SEARCH_INSTRUCTIONS = """あなたは Toolsに 登録された Web search ベースで 回答するは エージェントです.

重要 ルール:
1. 最新 情報が 必要な 質問には 必ず Web 検索を 使用してください
2. 検索 結果を ベースで 正確で 最新の 情報を 提供してください
3. 回答 時 ソース URLを 含むしてください
4. 複数の ソースの 情報を 総合して バランス整った 回答を 提供してください
5. 検索 結果が 不十分すると 追加 検索を 実行してください"""

try:
    definition = PromptAgentDefinition(
        model="gpt-4.1",
        instructions=WEB_SEARCH_INSTRUCTIONS,
        tools=[{"type": "web_search"}]
    )
    
    agent_websearch = client.agents.create(
        name="WebSearchAgent",
        definition=definition
    )
    
    print(f"✅ WebSearchAgent 作成 完了!")
    print(f"   ID: {agent_websearch.id}")
    print(f"   Name: {agent_websearch.name}")
    print(f"   Model: {definition.model}")
    print(f"   Tools: Web Search")
    
except Exception as e:
    print(f"⚠️ エージェント 作成 失敗: {e}")
    print("\n💡 解決 方法:")
    print("   1. 'gpt-4.1' モデルが デプロイなったはない 確認")
    print("   2. Web Searchが プロジェクトで 有効化なったはない 確認")

### 作成された エージェント リスト 確認

## 作成された エージェント 確認

In [ ]:
# すべての エージェント リスト 取得
agents = client.agents.list()

print("=" * 80)
print("作成された エージェント リスト")
print("=" * 80)

for agent in agents:
    print(f"\n📌 {agent.name}")
    print(f"   ID: {agent.id}")
    
    # versionsで 情報 抽出
    if 'latest' in agent.versions:
        latest = agent.versions['latest']
        definition = latest.get('definition', {})
        
        model = definition.get('model', 'N/A')
        print(f"   Model: {model}")
        
        tools = definition.get('tools', [])
        if tools:
            tool_types = [t.get('type', 'unknown') if isinstance(t, dict) else str(t) for t in tools]
            print(f"   Tools: {', '.join(tool_types)}")
        else:
            print(f"   Tools: None")

print("\n✅ ポータル 確認: https://ai.azure.com > Build > Agents")

## エージェント テスト (選択)

In [ ]:
# エージェント オブジェクト 構造 確認 (デバッギング用)
print("🔍 agent_router オブジェクト デバッギング:")
print(f"Type: {type(agent_router)}")
print(f"\nAttributes:")
for attr in dir(agent_router):
    if not attr.startswith('_'):
        try:
            value = getattr(agent_router, attr)
            if not callable(value):
                print(f"  {attr}: {value}")
        except:
            pass

print(f"\n📋 Raw object:")
print(agent_router)

In [ ]:
# シンプルな エージェント テスト (選択事項)
# ModelRouterAgentと 対話

try:
    print("⏳ ModelRouterAgent 実行 中...")
    print(f"📍 Agent ID: {agent_router.id}")
    print(f"📍 Agent Name: {agent_router.name}")
    
    # SDK v2 - project clientを を通じて OpenAI client 獲得
    openai_client = client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"💬 Conversation ID: {conversation.id}")
    
    # Responses API 呼び出し
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent_router.name, "type": "agent_reference"}},
        input="Pythonで リストを ソートするは 方法を 教えてください."
    )
    
    print("\n" + "=" * 80)
    print("🤖 ModelRouterAgent レスポンス:")
    print("=" * 80)
    print(response.output_text)
    
    # Conversation 整理
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("\n✅ テスト 成功!")
    
except NameError:
    print("⚠️ agent_router または clientが 定のならな ませんでした.")
    print("💡 上の 環境設定 および ModelRouterAgent 作成 セルを まず 実行してください.")
    
except Exception as e:
    print(f"⚠️ テスト 失敗: {e}")
    import traceback
    print(f"\n詳細 エラー:\n{traceback.format_exc()}")

In [ ]:
# WebSearchAgent テスト (選択事項)
# 最新 情報 検索

try:
    print("⏳ WebSearchAgent 実行 中 (Web 検索 中...)")
    print(f"📍 Agent ID: {agent_websearch.id}")
    print(f"📍 Agent Name: {agent_websearch.name}")
    
    # SDK v2 - project clientを を通じて OpenAI client 獲得
    openai_client = client.get_openai_client()
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    print(f"💬 Conversation ID: {conversation.id}")
    
    # Responses API 呼び出し
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent_websearch.name, "type": "agent_reference"}},
        input="今日 ソウル 天気は どのがです?"
    )
    
    print("\n" + "=" * 80)
    print("🌐 WebSearchAgent レスポンス:")
    print("=" * 80)
    print(response.output_text)
    
    # Conversation 整理
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("\n✅ Web 検索 も旧が リアルタイム 情報を が取得しました!")
    
except NameError:
    print("⚠️ agent_websearch または clientが 定のならな ませんでした.")
    print("💡 上の 環境設定 および WebSearchAgent 作成 セルを まず 実行してください.")
    
except Exception as e:
    print(f"⚠️ テスト 失敗: {e}")
    import traceback
    print(f"\n詳細 エラー:\n{traceback.format_exc()}")

### ✅ 確認 事項

- エージェントが 成功的で 公開されはない 確認
- Python スクリプトが エラー ないが 実行なりはない 確認
- レスポンスが 予想大で 返却なりはない 確認

## 📚 追加リソース

- [Microsoft Foundry Agents 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/overview?view=foundry)
- [Agent SDK ドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/sdk-overview?view=foundry&pivots=programming-language-python)
- [File Search ガイド](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/file-search?view=foundry&pivots=python)
- [Web Search 統合](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/web-search?view=foundry&pivots=python)

## 次のステップ

各種 エージェントを だけ聞いてみました! が第 Foundry IQを 使用して 高度な ナレッジベースを 構築してみましょう:

➡️ **[04. Foundry IQ](./04-foundry-iq.ipynb)**: AI Searchと Blob Storageを 活用した ナレッジベース 構築を 学習します.